<a href="https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/03_vision_foundation_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Maritime Vision Applications
## Session 1 · Part 1 — Vision Foundation Models: the CNN era
*Torben Globisch · VACOT*

---

| Notebook | Topic |
|---|---|
| 00 | Artificial Neural Networks: neurons, layers, learning |
| 01 | Computer Vision & an MLP on MNIST |
| 02 | Convolutional Neural Networks |
| **→ 03** | **Vision Foundation Models: the CNN era** |
| 04 | Vision Transformers |
| 05 | Self-supervised pretraining & transfer |

**Session goal.** Starting from the convolution operation you already know, we build up to the two dominant families
of Vision Foundation Models: **CNN-based** (ResNet, EfficientNet, …, *this notebook*) and **Transformer-based** (ViT, DINOv2, …, *notebooks 04 & 05*).

**By the end of this notebook you can**
1. show that kernels can be **learned** from input–target pairs instead of hand-crafted,
2. define a Vision Foundation Model (VFM) and its three-phase lifecycle,
3. explain why **ImageNet** changed everything, and why **skip connections** made very deep CNNs possible,
4. use a pretrained ResNet as a **feature extractor** and evaluate it with a **linear probe**.

> 💡 Use a GPU runtime: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# @title ⚙️ Setup: imports, colours & helpers — run me first { display-mode: "form" }
import math, io, time, urllib.request
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from IPython.display import HTML, display
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# Colour palette — the same colours as the lecture slides
NAVY, BLUE, LAVENDER, ORANGE, GREEN, RED, GREY, LIGHT = (
    "#2D3A8C", "#5B67B8", "#A3A9DB", "#E8702A", "#3E9E64", "#C0392B", "#8A8FA8", "#EEF0F7")
plt.rcParams.update({
    "figure.dpi": 100, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlecolor": NAVY,
    "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, GREEN, BLUE, RED, GREY]),
    "animation.html": "jshtml", "animation.embed_limit": 80,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU")

def play(anim):
    """Show a matplotlib animation as an HTML player (play / pause / step frame by frame)."""
    plt.close(anim._fig)
    return HTML(anim.to_jshtml())

# Maritime example photos from the COCO validation set (Flickr, Creative Commons licences).
MARITIME = {"trawler": 330369, "sailboat": 286422, "lifeboat": 400803, "marina": 442822,
            "fishing_boat": 451090, "yacht": 489014, "ferry": 543300}

def load_image(name, size=None):
    """Load one of the MARITIME photos (or any image URL) as an RGB PIL image."""
    url = f"http://images.cocodataset.org/val2017/{MARITIME[name]:012d}.jpg" if name in MARITIME else name
    with urllib.request.urlopen(url, timeout=30) as r:
        img = Image.open(io.BytesIO(r.read())).convert("RGB")
    if size is not None:  # centre-crop to a square, then resize
        s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
        img = img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC)
    return img

---
# 1 · Recap: convolution, and how kernels are learned

In notebook 02 we slid the hand-crafted **Sobel-X** kernel over an image and got a bright stripe where the vertical edge was:

$$ (I * K)_{i,j} = \sum_{m}\sum_{n} I_{i+m,\,j+n} \cdot K_{m,n}, \qquad K_\text{Sobel-X} = \begin{pmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{pmatrix} $$

Three key properties: **local connectivity**, **parameter sharing**, **translation equivariance**.

## 1.1 · Learning a kernel from input–target pairs
In a real CNN nobody writes down the kernel. Instead we provide **input–target pairs** $(I, T)$ and let gradient descent
find the kernel parameters $\theta$:

$$ \mathcal{L}(\theta) = \sum_{i,j} \big( (I * K_\theta)_{i,j} - T_{i,j} \big)^2, \qquad K_\theta \leftarrow K_\theta - \eta\, \frac{\partial \mathcal{L}}{\partial K_\theta} $$

**Experiment:** we create targets $T$ by applying a "secret" kernel to some boat photos, start from a **random**
$K_\theta$ and check whether gradient descent recovers the secret.

In [ ]:
SOBEL_X = [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]
SOBEL_Y = [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]
BLUR = [[1 / 9] * 3] * 3

SECRET_KERNEL = SOBEL_X     # ✏️ try SOBEL_Y, BLUR, or your own 3×3 kernel!

images = torch.stack([torch.tensor(np.asarray(load_image(n, size=96).convert("L")), dtype=torch.float32) / 255
                      for n in ["trawler", "sailboat", "lifeboat", "ferry"]])[:, None]      # (4, 1, 96, 96)
K_secret = torch.tensor(SECRET_KERNEL, dtype=torch.float32)[None, None]
targets = F.conv2d(images, K_secret)                                                        # T = I * K_secret

torch.manual_seed(3)
K_theta = (0.5 * torch.randn(1, 1, 3, 3)).requires_grad_()        # random initialisation
opt = torch.optim.Adam([K_theta], lr=0.05)
history, losses = [], []
for step in range(400):
    loss = ((F.conv2d(images, K_theta) - targets) ** 2).mean()     # MSE between prediction and target
    opt.zero_grad(); loss.backward(); opt.step()
    history.append(K_theta.detach()[0, 0].clone()); losses.append(loss.item())

print("learned kernel:\n", history[-1].numpy().round(2))
print("secret kernel:\n", K_secret[0, 0].numpy())

In [ ]:
# @title 🎞️ Animation: a random kernel turns into an edge detector { display-mode: "form" }
fig, axes = plt.subplots(1, 5, figsize=(17, 3.6), gridspec_kw=dict(width_ratios=[1, 1, 1.2, 1.2, 1.4]))
vmax = max(1, np.abs(SECRET_KERNEL).max())
def kernel_panel(ax, k, title):
    im = ax.imshow(k, cmap="coolwarm", vmin=-vmax, vmax=vmax); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)
    txt = [[ax.text(c, r, "", ha="center", va="center", fontsize=11, fontweight="bold") for c in range(3)] for r in range(3)]
    return im, txt
im_k, txt_k = kernel_panel(axes[0], history[0], "learned $K_\\theta$")
_, txt_s = kernel_panel(axes[1], K_secret[0, 0], "secret kernel")
for (r, c), v in np.ndenumerate(K_secret[0, 0].numpy()): txt_s[r][c].set_text(f"{v:.2g}")
with torch.no_grad(): preds = [F.conv2d(images[:1], h[None, None])[0, 0] for h in history]
rng = targets[0, 0].abs().max()
im_p = axes[2].imshow(preds[0], cmap="gray", vmin=-rng, vmax=rng); axes[2].set_title("result $I * K_\\theta$"); axes[2].axis("off")
axes[3].imshow(targets[0, 0], cmap="gray", vmin=-rng, vmax=rng); axes[3].set_title("target $T$"); axes[3].axis("off")
axes[4].plot(losses, color=LAVENDER, lw=1); dot, = axes[4].plot([], [], "o", color=ORANGE)
axes[4].set(yscale="log", xlabel="gradient step", title="loss $\\mathcal{L}(\\theta)$")
step_txt = fig.suptitle("", color=NAVY, fontweight="bold"); plt.tight_layout()

def update(f):
    t = min(f * 8, len(history) - 1); k = history[t]
    im_k.set_data(k)
    for (r, c), v in np.ndenumerate(k.numpy()): txt_k[r][c].set_text(f"{v:.1f}")
    im_p.set_data(preds[t]); dot.set_data([t], [losses[t]]); step_txt.set_text(f"step {t}")
    return [im_k, im_p, dot]

play(animation.FuncAnimation(fig, update, frames=51, interval=120))

**Key idea.** We never told the optimiser what the kernel should look like. We only said **what output we want**.
Gradient descent found the kernel values automatically.

In a trained CNN the targets are not edge maps but class labels, so the first-layer kernels are free to become whatever
helps classification. Empirically they very often turn into **edge and colour detectors**, Sobel-like filters
(remember the ResNet-18 kernels at the end of notebook 02). Edge detectors *emerge from data*.

## 1.2 · The special case stride = kernel size: patches
Remember the output-size formula $\lfloor (W-K+2p)/s \rfloor + 1$. When the stride equals the kernel size ($s = K$),
the kernel positions **never overlap**: the image is cut into a grid of non-overlapping **patches**, and every patch
becomes exactly one output value.

In [ ]:
img224 = load_image("trawler", size=224)
x = torch.tensor(np.asarray(img224), dtype=torch.float32).permute(2, 0, 1)[None] / 255   # (1, 3, 224, 224)
patchify = nn.Conv2d(3, 768, kernel_size=16, stride=16)
print("input:", tuple(x.shape), " → conv(kernel 16, stride 16) →", tuple(patchify(x).shape), "= a 14 × 14 grid of patches")

fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw=dict(width_ratios=[1, 1.25]))
axes[0].imshow(img224)
for k in range(0, 225, 16): axes[0].axhline(k - .5, color="white", lw=.8); axes[0].axvline(k - .5, color="white", lw=.8)
axes[0].set_title("224 × 224 image, 16 × 16 patches → 14 × 14 = 196 patches"); axes[0].axis("off")
axes[1].axis("off"); axes[1].set_title("…unrolled into a sequence of 196 'tokens'")
P = x[0].unfold(1, 16, 16).unfold(2, 16, 16).permute(1, 2, 0, 3, 4).reshape(196, 3, 16, 16)
for k in range(196):
    r, c = divmod(k, 20)
    axes[1].imshow(P[k].permute(1, 2, 0), extent=[c * 18, c * 18 + 16, -r * 18 - 16, -r * 18])
axes[1].set_xlim(-2, 20 * 18); axes[1].set_ylim(-10 * 18 - 2, 2)
plt.tight_layout(); plt.show()

Hold on to this picture. It is **exactly** how a Vision Transformer reads an image (notebook 04).

---
# 2 · What is a Vision Foundation Model?

> **Working definition.** A Vision Foundation Model (VFM) is a neural network that
> 1. is **pretrained** on a **large dataset** using a **pretext task**,
> 2. learns **general visual representations**,
> 3. can be used as a starting point for **many downstream tasks** with minimal adaptation.

Pretext task examples: classify the 1000 ImageNet categories (*supervised*), predict missing image patches (*MAE*),
match different crops of the same image (*DINO*).

The definition is intentionally fuzzy. Rather than asking *"is this a foundation model?"*, ask: **"how foundational is it?"**

In [ ]:
# @title 🎨 Figure: the three phases of a foundation model { display-mode: "form" }
fig, ax = plt.subplots(figsize=(15, 4.4)); ax.axis("off"); ax.set_xlim(0, 15); ax.set_ylim(0, 4.6)
phases = [
    ("1. Pretraining", NAVY, ["large dataset", "pretext task", "often no labels"], "ImageNet · LVD-142M\nMAE masking · DINO",
     "The model solves a 'puzzle'\n(predict missing patches,\nmatch crops, classify …)"),
    ("2. Benchmarking", ORANGE, ["frozen backbone", "linear probe / k-NN", "representation quality"], "ImageNet top-1\nADE20k mIoU · COCO AP",
     "How good are the frozen\nfeatures? Separates pretraining\nfrom fine-tuning."),
    ("3. Application", GREEN, ["fine-tune or adapter", "task-specific head", "deploy to real domain"], "maritime detection\nmedical · satellite",
     "Adapt to your task: as simple\nas a linear head on frozen\nfeatures."),
]
for i, (title, col, bullets, ex, note) in enumerate(phases):
    x0 = 0.2 + i * 5
    ax.add_patch(FancyBboxPatch((x0, 1.75), 4.3, 2.7, boxstyle="round,pad=0.05", fc=col, ec=col, alpha=.12))
    ax.add_patch(FancyBboxPatch((x0, 3.85), 4.3, 0.6, boxstyle="round,pad=0.05", fc=col, ec=col))
    ax.text(x0 + 2.15, 4.15, title, ha="center", va="center", color="white", fontweight="bold", fontsize=13)
    ax.text(x0 + 0.2, 3.55, "\n".join("• " + b for b in bullets), va="top", fontsize=10.5, color=NAVY, linespacing=1.5)
    ax.text(x0 + 0.2, 2.35, ex, va="top", fontsize=9, color=col, style="italic")
    ax.text(x0 + 2.15, 1.45, note, ha="center", va="top", fontsize=9.5, color="black")
    if i < 2: ax.annotate("", xy=(x0 + 4.95, 3.1), xytext=(x0 + 4.4, 3.1), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=2))
plt.show()

---
# 3 · ImageNet: the dataset that changed everything

**ImageNet-1k** (ILSVRC, the *ImageNet Large Scale Visual Recognition Challenge*, 2010–2017):
1.28 million training images · 1000 categories · 50,000 validation images.

**2012, the breakthrough:** *AlexNet* (Krizhevsky, Sutskever & Hinton) cut the top-5 error from 26.2 % (2nd place) to
15.3 %. It was the first time a CNN won, and it won by a huge margin. From then on, every winner was a CNN.

In [ ]:
ilsvrc = [(2010, 28.2, "NEC-UIUC", False), (2011, 25.8, "XRCE", False), (2012, 15.3, "AlexNet", True), (2013, 11.7, "ZFNet / Clarifai", True),
          (2014, 6.7, "GoogLeNet", True), (2015, 3.57, "ResNet", True), (2016, 2.99, "Trimps-Soushen", True), (2017, 2.25, "SENet", True)]
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar([str(y) for y, *_ in ilsvrc], [e for _, e, *_ in ilsvrc], color=[NAVY if cnn else GREY for *_, cnn in ilsvrc])
for b, (_, e, name, _) in zip(bars, ilsvrc):
    ax.text(b.get_x() + b.get_width() / 2, e + 0.6, f"{e}%\n{name}", ha="center", fontsize=8.5)
ax.axhline(5.1, color=ORANGE, ls="--", lw=2); ax.text(7.45, 9.0, "human ≈ 5.1 % (dashed)", color=ORANGE, ha="right", fontweight="bold")
ax.set(ylabel="top-5 error (%)", title="ILSVRC winners: top-5 classification error (grey = no CNN)", ylim=(0, 33))
plt.show()

*Top-5 error:* the true class is not among the model's 5 best guesses. The human number comes from Andrej Karpathy
labelling the test set himself in 2014.

## 3.1 · CNN architecture evolution
Each generation introduced **one key architectural idea**. The numbers below come straight from `torchvision`'s
model zoo: accuracy of torchvision's reference training of each architecture (single crop, ImageNet-1k validation top-1).

In [ ]:
from torchvision import models as tvm
zoo = [("AlexNet", 2012, tvm.AlexNet_Weights.IMAGENET1K_V1, "first GPU-trained deep CNN, ReLU, dropout"),
       ("VGG-16", 2014, tvm.VGG16_Weights.IMAGENET1K_V1, "deeper: only 3×3 convolutions"),
       ("GoogLeNet", 2014, tvm.GoogLeNet_Weights.IMAGENET1K_V1, "inception modules, cheap"),
       ("ResNet-50", 2015, tvm.ResNet50_Weights.IMAGENET1K_V1, "skip connections → 100+ layers"),
       ("ResNet-152", 2015, tvm.ResNet152_Weights.IMAGENET1K_V1, ""),
       ("DenseNet-201", 2017, tvm.DenseNet201_Weights.IMAGENET1K_V1, "dense connections"),
       ("EfficientNet-B0", 2019, tvm.EfficientNet_B0_Weights.IMAGENET1K_V1, "compound scaling of depth/width/resolution"),
       ("EfficientNet-B7", 2019, tvm.EfficientNet_B7_Weights.IMAGENET1K_V1, ""),
       ("ConvNeXt-L", 2022, tvm.ConvNeXt_Large_Weights.IMAGENET1K_V1, "a CNN modernised with ViT tricks"),
       ("ViT-B/16", 2021, tvm.ViT_B_16_Weights.IMAGENET1K_V1, "no convolutions: a Transformer (nb. 04)")]
print(f"{'model':<17}{'year':<6}{'params':>9}{'GFLOPs':>8}{'top-1':>8}   key idea")
for name, year, w, idea in zoo:
    m = w.meta
    print(f"{name:<17}{year:<6}{m['num_params'] / 1e6:>8.1f}M{m['_ops']:>8.1f}{m['_metrics']['ImageNet-1K']['acc@1']:>7.1f}%   {idea}")

fig, ax = plt.subplots(figsize=(10, 4.6))
for name, year, w, idea in zoo:
    m = w.meta; acc = m["_metrics"]["ImageNet-1K"]["acc@1"]
    col = ORANGE if "ViT" in name else (GREEN if "ResNet" in name else NAVY)
    ax.scatter(m["_ops"], acc, s=m["num_params"] / 4e5, color=col, alpha=.6, ec="k")
    dx, dy = {"ConvNeXt-L": (-8, 8), "EfficientNet-B7": (8, -12), "DenseNet-201": (6, 4), "ResNet-50": (6, -10)}.get(name, (6, -3))
    ax.annotate(f"{name} ({year})", (m["_ops"], acc), xytext=(dx, dy), textcoords="offset points", fontsize=9, ha="right" if dx < 0 else "left")
ax.set(xscale="log", xlabel="compute per image (GFLOPs, log scale)", ylabel="ImageNet top-1 accuracy (%)",
       title="Accuracy vs. compute (bubble area ∝ number of parameters)")
plt.show()

Bigger is not automatically better: VGG-16 has 138 M parameters, but ResNet-50 is more accurate with 25 M,
and EfficientNet-B0 almost matches it with 5 M. **Architecture ideas matter.**

## 3.2 · Why skip connections matter
Before ResNet, simply stacking more layers made networks **worse**, even on the training set
(the *degradation problem*). ResNet's fix is tiny: let every block learn a **residual** $F(x)$ and add the input back:

$$ \text{output} = \text{ReLU}\big(F(x) + x\big) $$

The gradient can then flow straight through the "+ x" path to early layers, and a block that has nothing to add can simply learn $F(x) \approx 0$.

In [ ]:
# @title 🎨 Figure: plain block vs. residual block { display-mode: "form" }
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, residual in zip(axes, [False, True]):
    ax.axis("off"); ax.set_xlim(0, 6); ax.set_ylim(0, 7.2)
    ax.text(3, 6.7, "x", ha="center", fontsize=14)
    for y, lbl, col in [(5.2, "Conv + BN + ReLU", ORANGE), (3.4, "Conv + BN", ORANGE)]:
        ax.add_patch(FancyBboxPatch((1.5, y), 3, 0.8, boxstyle="round,pad=0.05", fc=col, alpha=.85, ec="none"))
        ax.text(3, y + 0.4, lbl, ha="center", va="center", color="white", fontweight="bold")
    ax.annotate("", xy=(3, 6.05), xytext=(3, 6.55), arrowprops=dict(arrowstyle="-|>", color=NAVY))
    ax.annotate("", xy=(3, 4.25), xytext=(3, 5.15), arrowprops=dict(arrowstyle="-|>", color=NAVY))
    if residual:
        ax.add_patch(Circle((3, 2.4), 0.3, fc="white", ec=GREEN, lw=2)); ax.text(3, 2.4, "+", ha="center", va="center", fontsize=16, color=GREEN)
        ax.annotate("", xy=(3, 2.72), xytext=(3, 3.35), arrowprops=dict(arrowstyle="-|>", color=NAVY))
        ax.plot([3, 5.3, 5.3], [6.45, 6.45, 2.4], color=GREEN, lw=2.5); ax.annotate("", xy=(3.32, 2.4), xytext=(5.3, 2.4), arrowprops=dict(arrowstyle="-|>", color=GREEN, lw=2.5))
        ax.text(5.45, 4.4, "skip: x", color=GREEN, fontweight="bold", rotation=90, va="center")
        ax.annotate("", xy=(3, 1.2), xytext=(3, 2.08), arrowprops=dict(arrowstyle="-|>", color=NAVY)); ax.text(3, 0.75, "ReLU( F(x) + x )", ha="center", fontsize=12)
        ax.set_title("Residual block (ResNet)", color=GREEN)
    else:
        ax.annotate("", xy=(3, 1.2), xytext=(3, 3.35), arrowprops=dict(arrowstyle="-|>", color=NAVY)); ax.text(3, 0.75, "ReLU( F(x) )", ha="center", fontsize=12)
        ax.set_title("Plain block")
plt.tight_layout(); plt.show()

### Experiment: plain vs. residual, shallow vs. deep
We build four small CNNs that differ **only** in depth (8 or 50 blocks) and in whether the blocks have a skip connection,
and train each one for 300 steps on MNIST. *(≈ 1 min on a T4 GPU.)*

In [ ]:
from torchvision import datasets
mnist = datasets.MNIST("data", train=True, download=True)
X_mnist = ((mnist.data.float() / 255) - 0.1307) / 0.3081
y_mnist = mnist.targets

class Block(nn.Module):
    def __init__(self, channels, residual):
        super().__init__()
        self.bn = nn.BatchNorm2d(channels)
        self.conv = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.residual = residual

    def forward(self, x):
        out = self.conv(F.relu(self.bn(x)))           # F(x)
        return x + out if self.residual else out       # the ONLY difference: "+ x"

class DeepCNN(nn.Module):
    def __init__(self, depth, residual, channels=16):
        super().__init__()
        self.stem = nn.Conv2d(1, channels, 3, stride=2, padding=1)
        self.blocks = nn.Sequential(*[Block(channels, residual) for _ in range(depth)])
        self.head = nn.Linear(channels, 10)

    def forward(self, x):
        x = self.blocks(F.relu(self.stem(x.unsqueeze(1))))
        return self.head(x.mean((2, 3)))               # global average pooling → classifier

runs = {}
for depth in [8, 50]:
    for residual in [False, True]:
        name = f"{'residual' if residual else 'plain'}, {depth} blocks"
        torch.manual_seed(0); model = DeepCNN(depth, residual).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        # gradient norm of every block at initialisation
        xb, yb = X_mnist[:128].to(device), y_mnist[:128].to(device)
        F.cross_entropy(model(xb), yb).backward()
        grads = [b.conv.weight.grad.norm().item() for b in model.blocks]; model.zero_grad()
        losses = []; t0 = time.time()
        for step in range(300):
            idx = torch.randint(0, len(X_mnist), (128,))
            loss = F.cross_entropy(model(X_mnist[idx].to(device)), y_mnist[idx].to(device))
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        runs[name] = (losses, grads)
        print(f"{name:<22} final loss {np.mean(losses[-30:]):.3f}   ({time.time() - t0:.0f}s)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
style = {"plain, 8 blocks": (GREY, "--"), "residual, 8 blocks": (GREEN, "--"), "plain, 50 blocks": (GREY, "-"), "residual, 50 blocks": (GREEN, "-")}
for name, (losses, grads) in runs.items():
    c, ls = style[name]
    axes[0].plot(np.convolve(losses, np.ones(15) / 15, mode="valid"), color=c, ls=ls, lw=2.2, label=name)
    if "50" in name: axes[1].plot(range(1, len(grads) + 1), grads, "o-", color=c, ms=3, label=name)
axes[0].set(xlabel="training step", ylabel="training loss (smoothed)", title="Training loss", yscale="log"); axes[0].legend()
axes[1].set(xlabel="block (1 = closest to the input)", ylabel="‖∂L/∂W‖ at initialisation", yscale="log", title="Gradient size per block (50 blocks)"); axes[1].legend()
plt.tight_layout(); plt.show()

* At depth 8 both versions train fine.
* At depth 50 the **plain** network trains much worse than its shallow version. More layers *hurt*, even on the training data.
  Its gradients are badly scaled: they differ by orders of magnitude from block to block.
* The **residual** network keeps the gradients balanced and trains well even at depth 50.

That's why **ResNet-50** (2015) became *the* default backbone for transfer learning for years.

---
# 4 · The CNN pretraining paradigm
**Train once, reuse everywhere:**

In [ ]:
# @title 🎨 Figure: pretrain once, reuse the backbone everywhere { display-mode: "form" }
fig, ax = plt.subplots(figsize=(14, 4.3)); ax.axis("off"); ax.set_xlim(0, 14); ax.set_ylim(0, 5)
def box(x, y, w, h, text, col, txtcol="white", fs=11):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.06", fc=col, ec="none"))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", color=txtcol, fontsize=fs, fontweight="bold")
box(4.5, 4.0, 5, 0.8, "Pretrain on ImageNet\n1.28 M images · 1000 classes · cross-entropy", NAVY, fs=10)
box(5.0, 2.55, 4, 0.8, "Backbone (ResNet-50)\nweights θ*", ORANGE, fs=10)
ax.annotate("", xy=(7, 3.4), xytext=(7, 3.95), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=2))
for i, (task, data) in enumerate([("Classification", "ImageNet labels"), ("Detection", "COCO boxes"), ("Segmentation", "COCO masks"), ("Depth estimation", "depth labels")]):
    x = 0.4 + i * 3.4
    box(x, 0.5, 2.8, 1.0, f"{task}\nfine-tuned on\n{data}", GREEN, fs=9.5)
    ax.annotate("", xy=(x + 1.4, 1.55), xytext=(7, 2.5), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=1.5))
plt.show()

Each downstream model is still fine-tuned on its own labelled data, so these are **task-specific** models built on a shared
backbone. Not yet very "foundational" in the broad sense, but the idea of a reusable pretrained backbone was born.

---
# 5 · Demo: a pretrained ResNet-50 as a feature extractor

**Plan**
1. load a ResNet-50 with ImageNet weights,
2. remove its final classification layer → every image becomes a **2048-dimensional feature vector**,
3. explore the feature space: which images end up close to each other?
4. **linear probe:** freeze the backbone and train only a linear layer on a *new* dataset (CIFAR-10),
   then compare with a CNN trained from scratch on the same (small) amount of labels.

*(Note: `pretrained=True` in older tutorials is deprecated; today you pass `weights=...`.)*

In [ ]:
import torchvision
from torchvision.models import ResNet50_Weights

weights = ResNet50_Weights.IMAGENET1K_V2
backbone = torchvision.models.resnet50(weights=weights)
backbone.fc = nn.Identity()                  # remove the final FC layer (2048 → 1000)
backbone = backbone.eval().to(device)
preprocess = weights.transforms()            # resize 232 → centre-crop 224 → normalise

with torch.no_grad():
    z = backbone(preprocess(load_image("trawler"))[None].to(device))
print("feature vector shape:", tuple(z.shape))

### A new dataset: CIFAR-10
10 classes of tiny 32 × 32 images, including **ships**. The backbone has never been trained on it.
To mimic a realistic low-label scenario (labelling maritime data is expensive!) we only use **5,000** labelled training images.

In [ ]:
from torchvision import datasets, transforms
cifar_train = datasets.CIFAR10("data", train=True, download=True)
cifar_test = datasets.CIFAR10("data", train=False, download=True)
CIFAR_CLASSES = cifar_train.classes

N_TRAIN, N_TEST = 5000, 1000
g = torch.Generator().manual_seed(0)
idx_train = torch.randperm(len(cifar_train), generator=g)[:N_TRAIN]
idx_test = torch.randperm(len(cifar_test), generator=g)[:N_TEST]
imgs_train = torch.tensor(cifar_train.data[idx_train]).permute(0, 3, 1, 2)   # uint8 (N, 3, 32, 32)
imgs_test = torch.tensor(cifar_test.data[idx_test]).permute(0, 3, 1, 2)
labels_train = torch.tensor(cifar_train.targets)[idx_train]
labels_test = torch.tensor(cifar_test.targets)[idx_test]

fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for c in range(10):
    for r, i in enumerate((labels_train == c).nonzero()[:2, 0]):
        axes[r, c].imshow(imgs_train[i].permute(1, 2, 0)); axes[r, c].axis("off")
    axes[0, c].set_title(CIFAR_CLASSES[c], color=ORANGE if CIFAR_CLASSES[c] == "ship" else NAVY)
plt.tight_layout(); plt.show()

In [ ]:
@torch.no_grad()
def extract_features(model, imgs_uint8, size=224, batch_size=200):
    """Upsample the tiny images to the backbone's input size, normalise, and run them through the frozen backbone."""
    mean = torch.tensor([0.485, 0.456, 0.406], device=device)[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225], device=device)[:, None, None]
    feats = []
    for i in range(0, len(imgs_uint8), batch_size):
        x = imgs_uint8[i:i + batch_size].to(device).float() / 255
        x = F.interpolate(x, size=size, mode="bilinear", align_corners=False)
        feats.append(model((x - mean) / std).cpu())
    return torch.cat(feats)

t0 = time.time()
feat_train = extract_features(backbone, imgs_train)
feat_test = extract_features(backbone, imgs_test)
print(f"features: train {tuple(feat_train.shape)}, test {tuple(feat_test.shape)}  ({time.time() - t0:.0f}s)")

### Explore the feature space
**t-SNE** squeezes the 2048-D vectors into 2-D while trying to keep neighbours together. The backbone never saw CIFAR-10
labels, yet images of the same class already cluster together:

In [ ]:
from sklearn.manifold import TSNE
emb = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(feat_test.numpy())
cmap10 = plt.get_cmap("tab10")
fig, ax = plt.subplots(figsize=(8.5, 6.5))
for c in range(10):
    m = labels_test == c
    ax.scatter(*emb[m].T, s=14 if CIFAR_CLASSES[c] != "ship" else 30, color=cmap10(c), label=CIFAR_CLASSES[c],
               ec="k" if CIFAR_CLASSES[c] == "ship" else "none", lw=.4, alpha=.85)
ax.legend(markerscale=2, bbox_to_anchor=(1, 1), loc="upper left"); ax.set_xticks([]); ax.set_yticks([])
ax.set_title("t-SNE of frozen ResNet-50 features (CIFAR-10 test images)"); plt.show()

**Image retrieval:** for a few query ships, find the most similar training images (cosine similarity of the features):

In [ ]:
fn = F.normalize(feat_train, dim=1); fq = F.normalize(feat_test, dim=1)
queries = (labels_test == CIFAR_CLASSES.index("ship")).nonzero()[:4, 0]
fig, axes = plt.subplots(len(queries), 7, figsize=(12, 7))
for r, q in enumerate(queries):
    sims = fn @ fq[q]; nn_idx = sims.topk(6).indices
    axes[r, 0].imshow(imgs_test[q].permute(1, 2, 0)); axes[r, 0].set_title("query", color=ORANGE, fontweight="bold")
    for c, j in enumerate(nn_idx):
        axes[r, c + 1].imshow(imgs_train[j].permute(1, 2, 0))
        axes[r, c + 1].set_title(f"{CIFAR_CLASSES[labels_train[j]]} ({sims[j]:.2f})", fontsize=9,
                                 color=GREEN if labels_train[j] == labels_test[q] else RED)
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("Nearest neighbours in feature space", color=NAVY, fontweight="bold"); plt.tight_layout(); plt.show()

### ✏️ Linear probe vs. training from scratch
**Linear probe:** the backbone stays frozen; we only train a single linear layer $\hat y = W z + b$ on the features
($2048 \times 10 + 10 = 20{,}490$ parameters).

Fill in the linear probe below. Training on pre-computed features is very fast.

In [ ]:
def train_linear_probe(feat_tr, y_tr, feat_te, y_te, epochs=100, lr=1e-3):
    probe = None   # TODO: a single nn.Linear from feature dimension to 10 classes
    opt = torch.optim.Adam(probe.parameters(), lr=lr, weight_decay=1e-4)
    loader = DataLoader(TensorDataset(feat_tr, y_tr), batch_size=256, shuffle=True)
    for epoch in range(epochs):
        for f, y in loader:
            loss = F.cross_entropy(probe(f), y)
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (probe(feat_te).argmax(1) == y_te).float().mean().item()

In [ ]:
# @title 💡 Solution { display-mode: "form" }
def train_linear_probe(feat_tr, y_tr, feat_te, y_te, epochs=100, lr=1e-3):
    probe = nn.Linear(feat_tr.shape[1], 10)
    opt = torch.optim.Adam(probe.parameters(), lr=lr, weight_decay=1e-4)
    loader = DataLoader(TensorDataset(feat_tr, y_tr), batch_size=256, shuffle=True)
    for epoch in range(epochs):
        for f, y in loader:
            loss = F.cross_entropy(probe(f), y)
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (probe(feat_te).argmax(1) == y_te).float().mean().item()

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0)
acc_probe = train_linear_probe(feat_train, labels_train, feat_test, labels_test)
print(f"linear probe on frozen ResNet-50 features: {acc_probe:.1%} test accuracy")

Now the baseline: a CNN trained **from scratch** on the very same 5,000 labelled images (with data augmentation, 30 epochs).

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(cin, cout): return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))
        self.net = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 10))
    def forward(self, x): return self.net(x)

def to_input(x_uint8):
    mean = torch.tensor([0.49, 0.48, 0.45], device=x_uint8.device)[:, None, None]
    return (x_uint8.float() / 255 - mean) / 0.25

torch.manual_seed(0)
scratch = SmallCNN().to(device); opt = torch.optim.Adam(scratch.parameters(), lr=2e-3)
loader = DataLoader(TensorDataset(imgs_train, labels_train), batch_size=128, shuffle=True)
augment = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()])
t0 = time.time()
for epoch in range(30):
    scratch.train()
    for x, y in loader:
        x = augment(x.to(device)); loss = F.cross_entropy(scratch(to_input(x)), y.to(device))
        opt.zero_grad(); loss.backward(); opt.step()
scratch.eval()
with torch.no_grad():
    acc_scratch = (scratch(to_input(imgs_test.to(device))).argmax(1).cpu() == labels_test).float().mean().item()
print(f"small CNN from scratch: {acc_scratch:.1%} test accuracy ({time.time() - t0:.0f}s)")

fig, ax = plt.subplots(figsize=(7, 3.6))
b = ax.bar(["CNN from scratch\n(all weights trained)", "linear probe on\nfrozen ResNet-50"], [acc_scratch, acc_probe], color=[GREY, NAVY], width=.55)
ax.bar_label(b, labels=[f"{acc_scratch:.1%}", f"{acc_probe:.1%}"], fontweight="bold")
ax.set_ylim(0, 1); ax.set_ylabel("CIFAR-10 test accuracy"); ax.set_title(f"Same {N_TRAIN:,} labelled images"); plt.show()

Training only a single linear layer on top of the frozen pretrained features beats a full CNN trained from scratch on the
same labels, by a wide margin. The knowledge from 1.28 M ImageNet images **transfers**. This is the core promise of foundation models.

### 🔬 Experiments
1. Change `N_TRAIN` to 500 or 20,000. How does the gap between the two approaches change?
2. Replace ResNet-50 by `torchvision.models.resnet18` (512-dim features). Smaller backbone → worse probe?
3. Use a *randomly initialised* ResNet-50 (`weights=None`) as the frozen backbone. What does that tell you?

---
# Summary
1. Kernels are **learned** from input–target pairs by gradient descent; edge detectors emerge from data.
2. Stride = kernel size gives **non-overlapping patches**: the bridge to Vision Transformers.
3. A **Vision Foundation Model** is pretrained on a large dataset with a pretext task and reused for many downstream tasks.
   Ask *how foundational* a model is, not whether it is one.
4. **ImageNet + CNNs** (AlexNet 2012 → ResNet 2015 → EfficientNet 2019) made pretrained backbones the default;
   **skip connections** made networks with 100+ layers trainable.
5. A frozen pretrained backbone + a **linear probe** beats training from scratch when labels are scarce.

**Next →** [04 · Vision Transformers](https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/04_vision_transformers.ipynb): replacing convolutions by attention.